In [4]:
# Import libraries
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'numpy'

In [ ]:
# Set project paths
PROJECT_ROOT = Path("..")
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "crop_yield_data.xlsx"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed"
FIGURES_PATH = PROJECT_ROOT / "reports" / "figures"

PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)
FIGURES_PATH.mkdir(parents=True, exist_ok=True)

### 1. Load Raw Data

In [ ]:
df_raw = pd.read_excel(RAW_DATA_PATH)

original_shape = df_raw.shape
df = df_raw.copy()

print(f"Original dataset shape: {original_shape}")
df.head()

The raw dataset was loaded from the `data/raw` folder. A copy was created so that the original dataset remained unchanged.

### 2. Inspect the dataset

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

In [ ]:
df.describe(include="all").T

The dataset was inspected to understand its dimensions, variables, data types and summary statistics before cleaning.

### 3. Clean column names

In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(r"[^a-z0-9_]", "", regex=True)
)

df.columns.tolist()

### 4. Check missing values

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": df.isna().mean() * 100
})

missing_summary

Explain

### 5. Check duplicates

In [ ]:
duplicate_count = df.duplicated().sum()
print(f"Duplicate rows: {duplicate_count}")

In [ ]:
if duplicate_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)

Explain

### 6. Check data types

In [ ]:
df.dtypes

In [ ]:
numeric_columns = [
    "rainfall",
    "temperature",
    "fertilizer",
    "nitrogen",
    "phosphorus",
    "potassium",
    "yield"
]

numeric_columns = [col for col in numeric_columns if col in df.columns]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [ ]:
df.dtypes

### 7. Standardise crop labels

In [ ]:
if "crop" in df.columns:
    df["crop"] = (
        df["crop"]
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
    )

    print(df["crop"].value_counts())

### 8. Check invalid values

In [ ]:
validation_summary = {}

for col in numeric_columns:
    validation_summary[col] = {
        "minimum": df[col].min(),
        "maximum": df[col].max(),
        "negative_values": int((df[col] < 0).sum())
    }

pd.DataFrame(validation_summary).T

In [ ]:
for col in ["rainfall", "fertilizer", "nitrogen", "phosphorus", "potassium", "yield"]:
    if col in df.columns:
        print(f"{col}: {(df[col] < 0).sum()} negative values")

Expalanation - Negative rainfall, fertiliser, nutrient or yield values would usually require investigation.

Temperature should not automatically be restricted to positive values without knowing the dataset context.

### 9. Outlier assessment

In [ ]:
def iqr_outlier_summary(data, columns):
    results = []

    for col in columns:
        q1 = data[col].quantile(0.25)
        q3 = data[col].quantile(0.75)
        iqr = q3 - q1

        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        outlier_count = (
            (data[col] < lower_bound) |
            (data[col] > upper_bound)
        ).sum()

        results.append({
            "variable": col,
            "lower_bound": lower_bound,
            "upper_bound": upper_bound,
            "outlier_count": int(outlier_count)
        })

    return pd.DataFrame(results)

In [ ]:
outlier_summary = iqr_outlier_summary(df, numeric_columns)
outlier_summary

### 10.Final data-quality check

In [ ]:
print(f"Original shape: {original_shape}")
print(f"Final shape: {df.shape}")
print(f"Remaining missing values: {df.isna().sum().sum()}")
print(f"Remaining duplicate rows: {df.duplicated().sum()}")

### 11. Save only the cleaned

In [ ]:
cleaned_file = PROCESSED_DATA_PATH / "crop_yield_cleaned.csv"

df.to_csv(cleaned_file, index=False)

print(f"Cleaned dataset saved to: {cleaned_file}")

### 12. Cleaning summary table

In [ ]:
cleaning_summary = pd.DataFrame({
    "item": [
        "Original rows",
        "Original columns",
        "Final rows",
        "Final columns",
        "Duplicate rows removed",
        "Remaining missing values"
    ],
    "value": [
        original_shape[0],
        original_shape[1],
        df.shape[0],
        df.shape[1],
        duplicate_count,
        int(df.isna().sum().sum())
    ]
})

cleaning_summary